# Unit 4, Lecture 3: The manager pattern and delegation

Last lecture, routing sent work to the right specialist using **rules you wrote**:
if the queue is billing, go to billing. Fast, free, predictable, and **blind** to
anything beyond its keywords.

Today the decision itself becomes an **agent**. A manager reads the request and,
in its own **judgement**, delegates to the right specialist, using the
agent-as-tool idea from Unit 3.

The honest core is a trade-off: rules are cheap and predictable but cannot read
intent; a manager reads intent but costs a call and can choose wrong.

The specialists and tool wiring build **offline**; the manager's actual decision
needs a lane, because deciding is what the model does.

## Specialists: narrow experts

Each specialist is an agent with a tight, single-domain instruction. A billing
specialist that only knows billing gives sharper billing answers than one
generalist juggling everything.

In [ ]:
from cse476.manager import build_specialists, SPECIALIST_INSTRUCTIONS
from agent_framework.openai import OpenAIChatClient
import os
from dotenv import load_dotenv
load_dotenv()

# construction is offline; a real key is only needed to RUN a specialist
client = OpenAIChatClient(model="x", api_key="dummy",
                          base_url="https://models.github.ai/inference")
specialists = build_specialists(client)
print("specialists:", list(specialists))
print()
print("billing instruction:", SPECIALIST_INSTRUCTIONS["billing"][:80], "...")

## Each specialist becomes a tool

`as_tool` wraps a specialist agent as something the manager can call. The
**description** is what the manager reads to decide who fits, an instruction, not
documentation.

In [ ]:
from cse476.manager import specialist_tools

tools = specialist_tools(specialists)
for t in tools:
    print(f"{t.name:16} -> {t.description}")

## The manager: an agent whose tools are agents

The manager plays the role L2's classifier played, deciding where work goes, but
by **judgement** instead of keywords, choosing among **specialist agents**
instead of plain functions. **This next cell needs a lane.**

In [ ]:
from cse476.manager import make_client, build_manager

manager = build_manager(make_client())
print("manager built:", manager.name)

# a clearly billing request: watch it delegate to the billing specialist
print(await manager.run("I was double charged this month, can you help?"))

## The case rules could not handle

Here is where the manager earns its cost: a messy request that spans domains and
matches no clean keyword rule. See which specialist it picks, and judge whether
that was a reasonable call.

In [ ]:
# billing? account? technical? a rule cannot tell. the manager reads intent.
print(await manager.run("the thing I paid for keeps logging me out"))

A keyword router would have to guess or drop this. The manager reads the whole
sentence and delegates by understanding. That is exactly what you are paying the
model call for, and also exactly where it might delegate **wrong** in a way a
rule never would.

## The trade-off, stated plainly

In [ ]:
from cse476.manager import MANAGER_MAP, rules_vs_manager

for concept, tie in MANAGER_MAP.items():
    print(f"{concept:20} ->  {tie}")
print()
for k, v in rules_vs_manager().items():
    print(f"{k:18}: {v}")

Neither is better in general. Use **rules** for clear, high-volume routing
where mistakes are cheap and speed matters. Use a **manager** for ambiguous
requests where getting the right expert is worth a call and a small risk of
error. Most mature systems do both: rules for the easy majority, a manager for
the hard cases.

## Your turn

**1. Build the manager.** Wire the three specialists as tools and build the
manager. Send a clearly billing request and confirm it delegates to billing.

**2. Feed it a messy request.** Send something spanning two domains, like "I paid
for a plan that now gives an error." See which specialist it picks. Was it
reasonable? A rule could not have made that call.

**3. Rules or manager?** For your capstone, name one routing decision that should
be a **rule** and one that should be a **manager**. Justify each in one sentence.
That judgement is the whole lecture.

In [ ]:
# your work here
